In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

llm = ChatOpenAI()

In [3]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explaination: str
    

In [4]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content
    return {'joke': response}

In [5]:
def generate_explaination(state:JokeState):
    prompt = f'write an explaination for the joke; - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explaination':response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explaination', generate_explaination)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explaination')
graph.add_edge('generate_explaination', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id":"1"}}

workflow.invoke({'topic':'AI'}, config=config1)

{'topic': 'AI',
 'joke': 'Why did the robot go to school?\n\nTo get a byte of knowledge!',
 'explaination': 'This joke plays on the double meaning of "byte" as both a unit of digital information and a homophone for "bite." The humor lies in the idea of a robot going to school not to learn in the traditional sense, but to literally consume a byte of knowledge. This unexpected twist adds a silly and playful element to the joke.'}

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'AI', 'joke': 'Why did the robot go to school?\n\nTo get a byte of knowledge!', 'explaination': 'This joke plays on the double meaning of "byte" as both a unit of digital information and a homophone for "bite." The humor lies in the idea of a robot going to school not to learn in the traditional sense, but to literally consume a byte of knowledge. This unexpected twist adds a silly and playful element to the joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f097c09-a393-6b67-8002-a621b5de7db0'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-09-22T14:29:52.544240+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f097c09-978a-6fc3-8001-23e3254ea4e4'}}, tasks=(), interrupts=())

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'AI', 'joke': 'Why did the robot go to school?\n\nTo get a byte of knowledge!', 'explaination': 'This joke plays on the double meaning of "byte" as both a unit of digital information and a homophone for "bite." The humor lies in the idea of a robot going to school not to learn in the traditional sense, but to literally consume a byte of knowledge. This unexpected twist adds a silly and playful element to the joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f097c09-a393-6b67-8002-a621b5de7db0'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-09-22T14:29:52.544240+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f097c09-978a-6fc3-8001-23e3254ea4e4'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'AI', 'joke': 'Why did the robot go to school?\n\nTo get a byte of knowledge!'}, next=('generate_explainati

In [11]:
config2 = {"configurable": {"thread_id":"2"}}

workflow.invoke({'topic':'Machine Learning'}, config=config2)

{'topic': 'Machine Learning',
 'joke': "Why did the machine learning algorithm go broke at the casino?\n\nBecause it kept trying to predict the outcome of roulette but couldn't handle the random variables!",
 'explaination': 'This joke plays on the idea that machine learning algorithms are designed to analyze patterns and make predictions based on historical data. However, in the case of roulette, the outcome is purely based on chance and random variables, making it impossible for the algorithm to accurately predict the outcome. As a result, the algorithm ends up losing money at the casino because it cannot adapt to the unpredictable nature of the game.'}

In [12]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Machine Learning', 'joke': "Why did the machine learning algorithm go broke at the casino?\n\nBecause it kept trying to predict the outcome of roulette but couldn't handle the random variables!", 'explaination': 'This joke plays on the idea that machine learning algorithms are designed to analyze patterns and make predictions based on historical data. However, in the case of roulette, the outcome is purely based on chance and random variables, making it impossible for the algorithm to accurately predict the outcome. As a result, the algorithm ends up losing money at the casino because it cannot adapt to the unpredictable nature of the game.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f097c0d-1230-69ff-8002-63cf46175a51'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '2'}, created_at='2025-09-22T14:31:24.673484+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_n